# Stage 1 vs. Stage 2 误差解耦分析
**Error Decoupling: Stage 1 Propagated vs. Stage 2 Introduced Errors**

## Six error categories
| 类型 | 定义 |
|------|------|
| **S1-FP-Para** | Stage 1把gold=0段落判为propaganda，Stage 2为其生成的所有span |
| **S1-FP-Tech** | 段落确实有propaganda，但Stage 1预测了不存在的技术 |
| **S2-FP** | Stage 1段落+技术均正确，但Stage 2定位到不存在/边界错误的span |
| **S1-FN-Para** | Stage 1完全漏报该段落，导致其中所有gold span不可达 |
| **S1-FN-Tech** | Stage 1识别了段落，但漏报了某种技术 |
| **S2-FN** | Stage 1段落+技术均正确，但Stage 2未能定位到gold span |

## 核心方法：从现有文件反推
利用Sup-FT的 `official_format.txt`（span格式）将span反映射到段落，重建Stage 1的段落级决策。

In [ ]:
# ============================================================
# CONFIGURATION — update these paths before running
# ============================================================
import os

# Root directory containing data/, results/, technique_list/, etc.
BASE = "your/path/here"  # e.g. "/home/user/propaganda-span-detection"

# SemEval-2023 / CheckThat!-2024 overlap evaluation set
# Expected subdirectories: en_overlap_articles/, po_overlap_articles/, ru_overlap_articles/
OVERLAP_DIR = "your/path/here"  # typically BASE + "/data_analy/overlap_analysis_results"

# CheckThat! 2024 Task 3 training data bundle
CHECKTHAT24_DIR = "your/path/here"  # root of the official CheckThat!-2024 data bundle

# S3 credentials (or set as environment variables)
os.environ.setdefault("AWS_ACCESS_KEY_ID",     "your-access-key-id")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")
S3_ENDPOINT = "https://your-s3-endpoint"
S3_BUCKET   = "your-s3-bucket"


## 0. 路径配置

In [ ]:
import re, boto3
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Tuple, Set

BASE = Path(BASE)

ARTICLE_DIRS = {
    'en': BASE / 'data_analy/overlap_analysis_results/en_overlap_articles/articles',
    'pl': BASE / 'data_analy/overlap_analysis_results/po_overlap_articles/articles',
    'ru': BASE / 'data_analy/overlap_analysis_results/ru_overlap_articles/articles',
}
GOLD_FILES = {
    'en': BASE / 'data_analy/overlap_analysis_results/en_overlap_articles/labels-subtask-3-spans.txt',
    'pl': BASE / 'data_analy/overlap_analysis_results/po_overlap_articles/labels-subtask-3-spans.txt',
    'ru': BASE / 'data_analy/overlap_analysis_results/ru_overlap_articles/labels-subtask-3-spans.txt',
}
# Stage1+S2A 联合输出（用于反推Stage1段落级决策）
STAGE1_FILES = {
    'supft': {
        'en': BASE / 'results/model_tests/semeval_task3_en_dev_predictions_official_format.txt',
        'pl': BASE / 'results/model_tests/semeval_task3_po_dev_predictions_official_format.txt',
        'ru': BASE / 'results/model_tests/semeval_task3_ru_dev_predictions_official_format.txt',
    },
    'prompta': {
        'en': BASE / 'results/LLM_tests/results_en_results_semeval_task3_dev_en_context_all.tsv',
        'pl': BASE / 'results/LLM_tests/semeval2023_task3_dev_results_po_po_context_all.tsv',
        'ru': BASE / 'results/LLM_tests/semeval2023_task3_dev_results_ru_ru_context_all.tsv',
    },
    'iterens': {
        'en': BASE / 'results/LLM_tests/results_en_results_semeval_task3_dev_en_voting_aggressive_all.tsv',
        'pl': BASE / 'results/LLM_tests/semeval2023_task3_dev_results_po_po_voting_aggressive_all.tsv',
        'ru': BASE / 'results/LLM_tests/semeval2023_task3_dev_results_ru_ru_voting_aggressive_all.tsv',
    },
}
S2C_LOCAL = BASE / 'results/s2c_predictions'
S2C_LOCAL.mkdir(parents=True, exist_ok=True)
S2C_FILES = {
    'en': S2C_LOCAL / 'predictions_en.tsv',
    'pl': S2C_LOCAL / 'predictions_pl.tsv',
    'ru': S2C_LOCAL / 'predictions_ru.tsv',
}
S3_ENDPOINT = "https://your-s3-endpoint"  # Set to your S3-compatible endpoint
S3_BUCKET   = "your-s3-bucket"
S3_PREFIX   = 'multi_label_models/exp1_s2c_mbert_l10_multilingual/'
OUTPUT_DIR  = BASE / 'results/error_decoupling'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PRIMARY = 'supft'

print('=== Path check ===')
for lang in ['en', 'pl', 'ru']:
    a  = ARTICLE_DIRS[lang].exists()
    g  = GOLD_FILES[lang].exists()
    s1 = STAGE1_FILES[PRIMARY][lang].exists()
    s2 = S2C_FILES[lang].exists()
    print(f'  [{lang.upper()}] articles={chr(9989) if a else chr(10060)}  '
          f'gold={chr(9989) if g else chr(10060)}  '
          f'stage1={chr(9989) if s1 else chr(10060)}  '
          f's2c={chr(9989) if s2 else "需下载"}')


## 1. 文件格式检测（先跑，确认列结构）

In [ ]:
def detect_format(filepath: Path, n=5):
    with open(filepath, encoding='utf-8') as f:
        lines = [f.readline().strip() for _ in range(n)]
    print(f'\n  文件: {filepath.name}')
    for i, ln in enumerate(lines[:3]):
        cols = ln.split('\t')
        print(f'  行{i+1} ({len(cols)}列): {repr(ln[:130])}')
    try:
        first = lines[0].split('\t')
        int(first[2]); int(first[3])
        print('  -> span format: article_id \\t tech \\t start \\t end  OK')
        return 'span'
    except (IndexError, ValueError):
        print('  -> 非标准span格式，需手动确认')
        return 'unknown'

print('=== Sup-FT 文件格式 ===')
for lang in ['en', 'pl', 'ru']:
    f = STAGE1_FILES['supft'][lang]
    if f.exists(): detect_format(f)

print('\n=== Prompt-A 文件格式（EN示例）===')
f_p = STAGE1_FILES['prompta']['en']
if f_p.exists(): detect_format(f_p)


## 2. 数据加载

In [ ]:
def download_s2c():
    s3 = boto3.client('s3', endpoint_url=S3_ENDPOINT)
    for lang in ['en', 'pl', 'ru']:
        local = S2C_FILES[lang]
        if local.exists():
            print(f'  [{lang.upper()}] 本地已有'); continue
        key = f'{S3_PREFIX}predictions_{lang}.tsv'
        print(f'  [{lang.upper()}] 下载 {key}...')
        s3.download_file(S3_BUCKET, key, str(local))
        print(f'  [{lang.upper()}] 完成')

download_s2c()


def nid(raw): return re.sub(r'^article', '', str(raw).strip())


def load_articles(d: Path):
    arts = {}
    for f in d.glob('*.txt'):
        arts[nid(f.stem)] = f.read_text(encoding='utf-8')
    print(f'    文章: {len(arts)} 篇')
    return arts


def split_para(text: str, lang: str):
    """切段落，返回 [(start_char, end_char, text)]"""
    delim = '\n' if lang == 'en' else '\n\n'
    result, pos = [], 0
    for part in text.split(delim):
        s, e = pos, pos + len(part)
        if part.strip(): result.append((s, e, part))
        pos = e + len(delim)
    return result


def load_spans(f: Path):
    """加载span格式文件: article_id \\t tech \\t start \\t end"""
    d = defaultdict(list)
    with open(f, encoding='utf-8') as fp:
        for line in fp:
            parts = line.strip().split('\t')
            if len(parts) < 4: continue
            try: d[nid(parts[0])].append((parts[1], int(parts[2]), int(parts[3])))
            except ValueError: continue
    total = sum(len(v) for v in d.values())
    print(f'    spans: {total} 条 / {len(d)} 篇')
    return dict(d)


def spans_to_para_level(arts, spans, lang):
    """将span级预测反映射为 {aid: {para_idx: {tech}}}"""
    res = defaultdict(lambda: defaultdict(set))
    for aid, sps in spans.items():
        if aid not in arts: continue
        paras = split_para(arts[aid], lang)
        for tech, s, e in sps:
            for pi, (ps, pe, _) in enumerate(paras):
                if ps <= s < pe: res[aid][pi].add(tech); break
    print(f'    Paragraph back-projection: {len(res)} 篇')
    return {k: dict(v) for k, v in res.items()}


def gold_to_para_level(arts, gold, lang):
    """将gold spans映射为 {aid: {pidx: {tech: [(s,e)]}}}"""
    res = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
    for aid, sps in gold.items():
        if aid not in arts: continue
        paras = split_para(arts[aid], lang)
        for tech, s, e in sps:
            for pi, (ps, pe, _) in enumerate(paras):
                if ps <= s < pe: res[aid][pi][tech].append((s, e)); break
    return {k: {pk: dict(pv) for pk, pv in v.items()} for k, v in res.items()}


print('Functions defined')


## 3. 解耦核心函数

In [ ]:
def iou(as_, ae, bs, be):
    i = max(as_, bs); j = min(ae, be)
    if i >= j: return 0.0
    inter = j - i
    union = (ae - as_) + (be - bs) - inter
    return inter / union if union else 0.0


@dataclass
class ErrRec:
    article_id: str
    para_idx: int
    technique: str
    error_type: str  # S1-FP-Para|S1-FP-Tech|S2-FP|S1-FN-Para|S1-FN-Tech|S2-FN
    span_start: int = -1
    span_end: int = -1
    best_iou: float = 0.0


def decouple(arts, gold_by_para, s1_by_para, s2_spans, lang, iou_thresh=0.5):
    # Group S2-C predictions by paragraph
    s2_by_para = {}
    for aid, sps in s2_spans.items():
        if aid not in arts: continue
        paras = split_para(arts[aid], lang)
        s2_by_para[aid] = defaultdict(lambda: defaultdict(list))
        for tech, s, e in sps:
            for pi, (ps, pe, _) in enumerate(paras):
                if ps <= s < pe: s2_by_para[aid][pi][tech].append((s, e)); break

    errors = []
    for aid in arts:
        gold_a = gold_by_para.get(aid, {})
        s1_a   = s1_by_para.get(aid, {})
        s2_a   = s2_by_para.get(aid, {})

        pairs = set()
        for pi, ts in s1_a.items():
            for t in ts: pairs.add((pi, t))
        for pi, td in gold_a.items():
            for t in td: pairs.add((pi, t))

        for pi, tech in pairs:
            s1_pred   = tech in s1_a.get(pi, set())
            gold_here = tech in gold_a.get(pi, {})

            # FP
            if s1_pred and not gold_here:
                etype = 'S1-FP-Tech' if len(gold_a.get(pi, {})) > 0 else 'S1-FP-Para'
                spans_here = s2_a.get(pi, {}).get(tech, [(-1, -1)])
                for s, e in spans_here:
                    errors.append(ErrRec(aid, pi, tech, etype, s, e))

            # FN
            elif not s1_pred and gold_here:
                etype = 'S1-FN-Tech' if len(s1_a.get(pi, set())) > 0 else 'S1-FN-Para'
                for gs, ge in gold_a[pi][tech]:
                    errors.append(ErrRec(aid, pi, tech, etype, gs, ge))

            # S1正确 -> 评估S2
            elif s1_pred and gold_here:
                gold_this = gold_a[pi][tech]
                s2_this   = s2_a.get(pi, {}).get(tech, [])
                matched   = set()
                for gs, ge in gold_this:
                    best, bi = 0.0, -1
                    for i, (ss, se) in enumerate(s2_this):
                        if i in matched: continue
                        v = iou(ss, se, gs, ge)
                        if v > best: best, bi = v, i
                    if best >= iou_thresh: matched.add(bi)
                    else: errors.append(ErrRec(aid, pi, tech, 'S2-FN', gs, ge, best))
                for i, (ss, se) in enumerate(s2_this):
                    if i not in matched:
                        errors.append(ErrRec(aid, pi, tech, 'S2-FP', ss, se))
    return errors


print('解耦Functions defined')


## 4. 主分析：Sup-FT × S2-C

In [ ]:
ERR_TYPES = ['S1-FP-Para','S1-FP-Tech','S2-FP','S1-FN-Para','S1-FN-Tech','S2-FN']
FP_TYPES  = ['S1-FP-Para','S1-FP-Tech','S2-FP']
FN_TYPES  = ['S1-FN-Para','S1-FN-Tech','S2-FN']
all_errors = {}

for lang in ['en', 'pl', 'ru']:
    print(f'\n{"="*50}  {lang.upper()}  {"="*50}')
    print('  加载文章...')
    arts = load_articles(ARTICLE_DIRS[lang])
    print('  加载gold spans...')
    gold = load_spans(GOLD_FILES[lang])
    print('  加载Sup-FT预测...')
    s1_spans = load_spans(STAGE1_FILES[PRIMARY][lang])
    s1_by_para = spans_to_para_level(arts, s1_spans, lang)
    print('  加载S2-C预测...')
    s2_spans = load_spans(S2C_FILES[lang])
    print('  构建gold段落索引...')
    gold_by_para = gold_to_para_level(arts, gold, lang)
    print('  运行解耦...')
    errors = decouple(arts, gold_by_para, s1_by_para, s2_spans, lang)
    all_errors[lang] = errors
    c = {et: sum(1 for e in errors if e.error_type == et) for et in ERR_TYPES}
    print(f'  FP: {sum(c[t] for t in FP_TYPES)}  FN: {sum(c[t] for t in FN_TYPES)}  Total: {len(errors)}')

print('\nAll done')


## 5. 汇总统计

In [ ]:
pct = lambda n, d: f'{100*n/d:.1f}%' if d else '-'

rows = []
for lang in ['en', 'pl', 'ru']:
    c = {et: sum(1 for e in all_errors[lang] if e.error_type == et) for et in ERR_TYPES}
    total_fp = sum(c[t] for t in FP_TYPES)
    total_fn = sum(c[t] for t in FN_TYPES)
    s1_fp = c['S1-FP-Para'] + c['S1-FP-Tech']
    s1_fn = c['S1-FN-Para'] + c['S1-FN-Tech']
    rows.append({'Lang': lang.upper(),
        'S1-FP-Para': c['S1-FP-Para'], 'S1-FP-Tech': c['S1-FP-Tech'], 'S2-FP': c['S2-FP'],
        'Total FP': total_fp, 'S1-FP%': pct(s1_fp,total_fp), 'S2-FP%': pct(c['S2-FP'],total_fp),
        'S1-FN-Para': c['S1-FN-Para'], 'S1-FN-Tech': c['S1-FN-Tech'], 'S2-FN': c['S2-FN'],
        'Total FN': total_fn, 'S1-FN%': pct(s1_fn,total_fn), 'S2-FN%': pct(c['S2-FN'],total_fn)})

df = pd.DataFrame(rows)
print('=== Error origin distribution (Sup-FT x S2-C) ===')
print(df.to_string(index=False))
df.to_csv(OUTPUT_DIR / 'decoupling_summary.csv', index=False)
print(f'Saved: {OUTPUT_DIR}/decoupling_summary.csv')


In [ ]:
# Per-technique distribution
for lang in ['en', 'pl', 'ru']:
    tc = defaultdict(lambda: defaultdict(int))
    for e in all_errors[lang]: tc[e.technique][e.error_type] += 1
    rows_t = []
    for tech, c in sorted(tc.items()):
        total_fp = sum(c[t] for t in FP_TYPES)
        total_fn = sum(c[t] for t in FN_TYPES)
        s1_fp = c['S1-FP-Para'] + c['S1-FP-Tech']
        rows_t.append({'Technique': tech,
            'FP_S1Para': c['S1-FP-Para'], 'FP_S1Tech': c['S1-FP-Tech'],
            'FP_S2': c['S2-FP'], 'FP_Total': total_fp,
            'FP_S1%': pct(s1_fp, total_fp), 'FP_S2%': pct(c['S2-FP'], total_fp),
            'FN_S1Para': c['S1-FN-Para'], 'FN_S1Tech': c['S1-FN-Tech'],
            'FN_S2': c['S2-FN'], 'FN_Total': total_fn})
    df_t = pd.DataFrame(rows_t).sort_values('FP_Total', ascending=False)
    out = OUTPUT_DIR / f'decoupling_by_tech_{lang}.csv'
    df_t.to_csv(out, index=False)
    print(f'\n=== {lang.upper()} Top-10 FP ===')
    print(df_t.head(10)[['Technique','FP_Total','FP_S1%','FP_S2%','FN_Total']].to_string(index=False))
    print(f'Saved: {out}')


## 6. 可视化

In [ ]:
COLORS = {'S1-FP-Para':'#c0392b','S1-FP-Tech':'#e67e22','S2-FP':'#f9ca74',
          'S1-FN-Para':'#2980b9','S1-FN-Tech':'#7fb3d3','S2-FN':'#a9cce3'}
LABELS = {'S1-FP-Para':'S1 Para-FP','S1-FP-Tech':'S1 Tech-FP','S2-FP':'S2-FP',
          'S1-FN-Para':'S1 Para-FN','S1-FN-Tech':'S1 Tech-FN','S2-FN':'S2-FN'}

langs = ['en','pl','ru']
x = np.arange(len(langs))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, grp, title in [
    (axes[0], FP_TYPES, 'False Positives: Origin Breakdown'),
    (axes[1], FN_TYPES, 'False Negatives: Origin Breakdown'),
]:
    bottoms = np.zeros(len(langs))
    totals = np.array([sum(1 for e in all_errors[l] if e.error_type in grp) for l in langs])
    for etype in grp:
        vals = np.array([sum(1 for e in all_errors[l] if e.error_type == etype) for l in langs])
        ax.bar(x, vals, 0.55, bottom=bottoms, color=COLORS[etype],
               label=LABELS[etype], edgecolor='white', linewidth=0.5)
        for i, (v, b, tot) in enumerate(zip(vals, bottoms, totals)):
            if tot > 0 and v/tot > 0.06:
                ax.text(x[i], b+v/2, f'{100*v/tot:.0f}%',
                        ha='center', va='center', fontsize=9, color='white', fontweight='bold')
        bottoms += vals
    ax.set_title(title, fontsize=12, pad=10)
    ax.set_xticks(x); ax.set_xticklabels([l.upper() for l in langs], fontsize=11)
    ax.set_ylabel('Count'); ax.legend(fontsize=9)
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Sup-FT x S2-C: Error Origin Decoupling', fontsize=13, fontweight='bold')
plt.tight_layout()
for ext in ['pdf','png']:
    fig.savefig(OUTPUT_DIR / f'fig_decoupling_overview.{ext}', bbox_inches='tight', dpi=200)
plt.close()
print(f'Saved figures to {OUTPUT_DIR}')


In [ ]:
# 气泡图：技术级 S1 vs S2 FP来源
for lang in ['en','pl','ru']:
    tc = defaultdict(lambda: defaultdict(int))
    for e in all_errors[lang]: tc[e.technique][e.error_type] += 1
    data = []
    for tech, c in tc.items():
        tot = sum(c[t] for t in FP_TYPES)
        if tot < 10: continue
        s1_r = (c['S1-FP-Para'] + c['S1-FP-Tech']) / tot
        data.append({'tech': tech.replace('_','\n'), 's1': s1_r, 's2': c['S2-FP']/tot, 'n': tot})
    if not data: continue
    fig, ax = plt.subplots(figsize=(9, 5))
    for d in data:
        col = '#c0392b' if d['s1'] > 0.5 else '#2980b9'
        ax.scatter(d['s1'], d['s2'], s=max(60, d['n']*0.3),
                   color=col, alpha=0.75, edgecolors='white', linewidth=1)
        ax.annotate(d['tech'], (d['s1'], d['s2']), fontsize=7,
                    ha='center', xytext=(0,8), textcoords='offset points')
    ax.axvline(0.5, color='gray', ls='--', alpha=0.5, lw=1)
    ax.axhline(0.5, color='gray', ls='--', alpha=0.5, lw=1)
    ax.set_xlabel('S1-Origin FP Ratio', fontsize=11)
    ax.set_ylabel('S2-Origin FP Ratio', fontsize=11)
    ax.set_title(f'{lang.upper()} - FP Origin by Technique', fontsize=12)
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.spines[['top','right']].set_visible(False)
    out = OUTPUT_DIR / f'fig_fp_bubble_{lang}.pdf'
    fig.savefig(out, bbox_inches='tight', dpi=200); plt.close()
    print(f'[{lang.upper()}] Saved: {out}')


## 7. 论文关键数字

In [ ]:
print('='*65)
print('  PAPER-READY NUMBERS  (Sup-FT x S2-C)')
print('='*65)
for lang in ['en','pl','ru']:
    c = {et: sum(1 for e in all_errors[lang] if e.error_type == et) for et in ERR_TYPES}
    total_fp = sum(c[t] for t in FP_TYPES)
    total_fn = sum(c[t] for t in FN_TYPES)
    s1_fp = c['S1-FP-Para'] + c['S1-FP-Tech']
    s1_fn = c['S1-FN-Para'] + c['S1-FN-Tech']
    print(f'\n  [{lang.upper()}]')
    print(f'    FP {total_fp} total')
    print(f'      S1-origin {s1_fp} ({pct(s1_fp,total_fp)})  para={c["S1-FP-Para"]}, tech={c["S1-FP-Tech"]}')
    print(f'      S2-origin {c["S2-FP"]} ({pct(c["S2-FP"],total_fp)})')
    print(f'    FN {total_fn} total')
    print(f'      S1-origin {s1_fn} ({pct(s1_fn,total_fn)})  para={c["S1-FN-Para"]}, tech={c["S1-FN-Tech"]}')
    print(f'      S2-origin {c["S2-FN"]} ({pct(c["S2-FN"],total_fn)})')
print('\n解读：')
print('  S1-FP高 -> Stage 1过度预测是主因，换更保守Stage 1可降FP')
print('  S2-FP高 -> S2置信阈值过低，提高threshold可改善')
print('  S1-FN-Para高 -> Stage 1段落召回是瓶颈，整篇漏报S2无法弥补')
print('  S2-FN高 -> Stage 1够好，S2定位能力是瓶颈')


## 8. 扩展：三种Stage 1方法对比（可选）

In [ ]:
# Uncomment to run
# ext_errors = {}
# for method in ['prompta', 'iterens']:
#     for lang in ['en', 'pl', 'ru']:
#         print(f'\n[{method} x {lang}]')
#         arts = load_articles(ARTICLE_DIRS[lang])
#         gold = load_spans(GOLD_FILES[lang])
#         s1_spans = load_spans(STAGE1_FILES[method][lang])
#         s1_by_para = spans_to_para_level(arts, s1_spans, lang)
#         s2_spans = load_spans(S2C_FILES[lang])
#         gold_by_para = gold_to_para_level(arts, gold, lang)
#         errors = decouple(arts, gold_by_para, s1_by_para, s2_spans, lang)
#         ext_errors[(method, lang)] = errors
#         c = {et: sum(1 for e in errors if e.error_type == et) for et in ERR_TYPES}
#         tot_fp = sum(c[t] for t in FP_TYPES)
#         s1_fp = c['S1-FP-Para'] + c['S1-FP-Tech']
#         print(f'  S1-FP: {pct(s1_fp,tot_fp)}  S2-FP: {pct(c["S2-FP"],tot_fp)}')
print('扩展分析就绪（默认注释）')
